Today’s core question:

> **When does repeating something become a design system?**

In earlier lessons, you built individual visual ideas. In this lesson, you will make one idea happen again and again. But not by boring copy-paste repetition. Nobody came here to press `Ctrl+C`, `Ctrl+V` until their keyboard asks for a vacation.

You will use **repetition with purpose**: a small rule that creates a larger artwork.

Let's stop and look at a classic abstract art painting for a second, Hilma af Klint’s **The Swan**,

![Hilma af Klint The Swan](painting.jpg)

Before going into the rules of a painting and how they're used, it's often good to take a step back and first consider,
well, 

<DoNow id="building-blocks">
What **are** the building blocks used here?

Don't just think, say it aloud!!
</DoNow>

![Hilma af Klint left and right](painting4.png)

The first thing to jump to our head is the blatant circle used many times in the painting no? But there's something special about this circle.
It's split into TWO colors. One color for the *left* of the circle, another on the *right*.

But wait!! The built in PyTamaro function we use to make circles is <API>ellipse</API> and that only takes **one** color into it's parameters. A TWO colored circle can't be possible? 

Well it is, but we're going to have to make our **own** `split_circle` function.

<DoNow id="work-around">
What should `split_circle` do?

(Hint: <API>circular_sector</API>). 
</DoNow>

That's right! Using semi circles (using `circular_sector`), we can make two halves of a circle (each with it's own color) and then glue them together. 

<Task>
**Complete** the function below.

Your function should:

1. create a left half using `left_color`,
2. create a right half using `right_color`,
3. place them beside each other,
4. return the combined graphic.
</Task>

In [ ]:
from pytamaro import (
    Graphic, Color,
    rectangle, beside, show_graphic, rgb_color,
    empty_graphic, circular_sector, rotate, overlay
)

# Colors
BACKGROUND_RED = rgb_color(170, 70, 50)
OFFWHITE       = rgb_color(220, 210, 195)
BLACK          = rgb_color(30, 30, 30)
BLUE           = rgb_color(95, 135, 185)
YELLOW         = rgb_color(230, 190, 80)
PINK           = rgb_color(220, 170, 165)

def split_circle(radius: float, left_color: Color, right_color: Color) -> Graphic:
    left_half = rotate(90, circular_sector(radius, 180, left_color))
    right_half = ...
    return ...

show_graphic(split_circle(100, BLACK, PINK))

<DoNow id="split-circle-abstraction">
What detail is hidden inside the function `split_circle`?

A. The exact rotations needed to build the two halves  
B. The background color of the final artwork  
C. The number of layers in the final artwork
</DoNow>

This is abstraction: once `split_circle` works, we can use the name without thinking about every sector and rotation again.

Great, we established our main building block, but the question now is, how is `split_circle` actually being used in **The Swan**? Enter...

## Repetition: Not Just “More Stuff”

Abstract artists often repeat shapes, colors, directions, or layers. But good repetition is rarely random. It can create:

- **rhythm**: your eye moves through the artwork,
- **sequence**: one element follows another,
- **visual echo**: a shape or color returns in a changed way.

<DoNow id="visual-rhythm">
Look back at the artwork image.

What repeats?

What changes each time it repeats?

Say it aloud!
</DoNow>

![Hilma af Klint repetition highlight](painting2.png)

As you can see, `split_circle` is repeatedly used 3 times, but the radius and color change. Given the layered look of the painting, we can think of the artist starting with the largest circle and then drawing smaller circles on top, moving inward by a *fixed* amount each time.

A good artist answer might be: “The `split_circle` shape repeats, but the size and colors change as the layers move inward.”

A good programmer answer might be: “That sounds like one reusable function, a changing number, and a list of color choices.”

Same observation. Two languages.


## Duplication vs. Repetition

There is a sneaky difference between **duplicating** and **repeating**.

**Duplication** means rewriting nearly the same thing again:

```python
large_layer = split_circle(150, OFFWHITE, BLUE)
medium_layer = split_circle(100, BLACK, YELLOW)
small_layer = split_circle(50, BLACK, PINK)
```

This works, but if the artwork grows to 20 layers, your code becomes spaghetti. 

**Repetition** means generating the repeated parts with a rule:

```python
for i in range(number_of_layers):
    ...
```

<DoNow id="duplication-or-repetition">
But what is the rule here?

(Hint: Look at the `large_layer, medium_layer, small_layer` example. What is the radius changing by as we move inward?)
</DoNow>

That's right! The radius of each split_circle layer decreases by a fixed value, in this case `50`, each time we add a layer.
We can also see this as starting from a given radius. Each layer is us subtracting a constant step from the previous layer:

```python
start_radius = 150

large_layer = 150
medium_layer = 150 - 50 = 100
small_layer = 100 - 50 = 50
```

When you spot a pattern like this, that is usually a **loop waiting to happen**. The code is basically waving at you like: “please stop copying me.”


## A Tiny Loop Preview

Let’s warm up with a loop that builds three text labels. No graphics yet, just the idea of a repeated rule.

<Task>
**Fill** out the missing code to make the loop iterate 3 times. Then answer this question: what stays the same each time, and what changes?
</Task>


In [ ]:
for i in range(...):
    layer_number = i + 1
    print("Build layer", layer_number)

The loop repeats the same instruction: print a layer message.

When we have a loop like this, how can we make the radius shrink with each iteration? We can make a variable called `current_radius`, use its current value, and then subtract the step after each repetition.

That is going to become important when we make layers of progressively smaller circles.

<DoNow id="loop-prediction">
Predict the output before running this in your head:

```python
current_radius = 40
step = 10

for i in range(4):
    print(current_radius)
    current_radius = current_radius - step
```

What numbers appear?
</DoNow>


<Task>
Now let's **make** our own little loop preview that produces the pattern we spotted (`150, 100, 50`).

Hint: subtract `radius_step` from the current radius in each iteration.
</Task>


In [ ]:
start_radius = 150
radius_step = 50
current_radius = start_radius

for i in range(...):
    print(current_radius)
    current_radius = ...


We updated `current_radius` with a rule. Nice.

But there is a new problem: every layer also needs **two colors**.

A loop can create the inward-changing sizes, but it still needs to know the color pair for each layer to build the `split_circle` inside the loop. Or else we'd be back to manually creating each layer!

That is where a list of tuples becomes useful.


## Why a List of Tuples?

A tuple can store one color pair:

```python
(OFFWHITE, BLUE)
```

A list can store the whole sequence of pairs, ordered from the outside inward:

```python
[(OFFWHITE, BLUE), (BLACK, YELLOW), (BLACK, PINK)]
```

In art terms, this is like a tiny visual score. Each entry says: “for this layer, use these two colors.”

<Task>
Complete the list of tuples for the three layers, from outermost circle to innermost circle.

Then print the number of layers and the second color pair.
</Task>


In [ ]:
color_pairs: list[tuple[Color, Color]] = [
    (OFFWHITE, BLUE),
    (..., ...),
    (..., ...),
]

print("Number of layers:", len(color_pairs))
print("Second color pair:", color_pairs[1])


<DoNow id="tuple-indexing-check">
If `pair = color_pairs[0]`, what do these expressions mean?

```python
pair[0]
pair[1]
```

Answer in art language, not just code language.
</DoNow>

A good answer: `pair[0]` is the left color of the outermost layer, and `pair[1]` is the right color of the outermost layer.


## Recreating *The Swan*

Now we combine the pieces:

- `split_circle` builds one reusable layer,
- `color_pairs` stores the layer colors from outside inward,
- a loop starts with the largest radius and subtracts `50` each time,
- each new, smaller `split_circle` is <API>overlay</API>-ed on top of the previous layers.

<Task>
Complete the loop-based composition.

Inside the loop, you need to:

1. get the current color pair,
2. extract `left_color` and `right_color`,
3. use `current_radius` as the layer radius,
4. overlay the new layer on top of `layers`,
5. subtract `radius_step` so the next layer is smaller.
</Task>


In [ ]:
def composition_fixed_step(size: float, color_pairs: list[tuple[Color, Color]]) -> Graphic:
    background = rectangle(size, size, BACKGROUND_RED)
    layers = empty_graphic()
    radius_step = 50
    current_radius = radius_step * len(color_pairs)

    for i in range(len(color_pairs)):
        pair = ...
        left_color = ...
        right_color = ...

        layer = split_circle(current_radius, ..., ...)
        layers = overlay(..., ...)
        current_radius = ...

    return overlay(layers, background)

show_graphic(composition_fixed_step(500, color_pairs))


## Why This Is Better Than Manual Code

<DoNow id="fixed-step-benefit">
What happens if you change the list `color_pairs`?

A. The loop automatically builds the new number of layers  
B. You must write a new `large_layer`, `medium_layer`, and `small_layer` manually  
C. The colors disappear and Python goes on holiday
</DoNow>

**A**. The list controls how many layers appear. That is a strong abstraction!

But we still have one design flaw.


## Stress Test: Add More Layers

We've managed to draw the painting, but a good design system should handle change. So let’s push our current version until it complains visually.

<Task>
Add at least **three more** color pairs to `many_color_pairs`.

Run the cell. What happens to the artwork when there are too many layers?
</Task>


In [ ]:
many_color_pairs: list[tuple[Color, Color]] = [
    (OFFWHITE, BLUE),
    (BLACK, YELLOW),
    (BLACK, PINK),
    (..., ...),
    (..., ...),
    (..., ...),
]

show_graphic(composition_fixed_step(500, many_color_pairs))


## The Problem With `radius_step = 50`

The loop works, and the list works. But the fixed inward step is still too rigid.

With three layers, the radii are:

```python
150, 100, 50
```

That fits nicely.

With many layers, the starting radius becomes much larger:

```python
300, 250, 200, 150, 100, 50, ...
```

In our `composition_fixed_step`, we define the starting value of our `current_radius` as `radius_step * len(color_pairs)`. 

If we have lots of layers in `color_pairs` the value `current_radius` may be so big, it's actually bigger than the canvas.

Our rule does not ask: “How much space do I actually have?”

This is where Lesson 3 comes back that taught us all about **proportion**.

Instead of saying “subtract 50 every time,” (which let's be honest is a number we *randomly* chose)

<DoNow id="even_better">
What would be **even** better?

Say it out loud!
</DoNow>

> The largest circle should occupy a chosen proportion of the canvas.

## Proportional Layering: Tie the Circles to the Canvas

We can do this by introducing a new paramater to our `composition` function, called `ratio`. 

If `ratio = 1`, the largest circle should occupy the whole canvas width. 

If `ratio = 0.6`, the largest circle should use only 60% of the canvas width. 

`ratio` answers one very specific question:

> **What fraction of the canvas width should the biggest circle use?**

<DoNow id="largest-diameter">
So if the canvas width is `size`, what do you think a good way of calculating the largest circle diameter would be?

Say it out loud!
</DoNow>


```python
largest_circle_diameter = size * ratio
```

We multiply because we are taking a **part of the canvas width**. `ratio` tells us how big a piece we take from the pie that is the canvas `size`.

![Hilma af Klint proportions](painting3.png)

<strong>Example:</strong><br />
    If <code>size = 500</code> and <code>ratio = 0.5</code>, the largest circle's diameter is:<br /><br />
    <code>500 × 0.5 = 250</code><br /><br />
    So the biggest circle is 250 pixels wide, leaving empty space around it.


Now we need the **radius**, because `split_circle(radius, ...)` asks for a radius, not a diameter.

A radius is half of a diameter, so:

```python
largest_circle_radius = largest_circle_diameter / 2
```

Finally, we do not want every circle to have the largest radius. We want the first circle to use the largest radius, and then each next circle should step inward evenly.

So we split the biggest radius into equal inward steps:

```python
radius_step = largest_circle_radius / number_of_circles
```

If the largest radius is `150` and there are `3` circles, each step is `50`. The loop starts with `current_radius = 150`, draws that layer, and then subtracts `50` after each layer:

```python
150, 100, 50
```

If there are `6` circles, the step becomes smaller automatically. The loop still follows the same update rule, but now subtracts `25` each time:

```python
150, 125, 100, 75, 50, 25
```

That is the whole idea: **same outer size, more circles means smaller inward steps, updated through `current_radius`.**

<DoNow id="proportional-growth-check">
If the canvas is `500` and `ratio` is `0.6`, what is the largest circle diameter?

What is the largest circle radius?

If there are `3` circles, what is `radius_step`?
</DoNow>


## Final Composition Function

Let's upgrade the composition so it adapts to the canvas and the number of layers.

<Task>
Complete the proportional version.

The main idea: do **not** hard-code the inward step. Compute it from the canvas `size`, the chosen `ratio`, and the number of circles.

hint: We can compute the `number_of_circles` using the `len(...)` function and the `color_pairs` list.
</Task>


In [ ]:
def composition(size: float, color_pairs: list[tuple[Color, Color]], ratio: float) -> Graphic:
    background = rectangle(size, size, BACKGROUND_RED)

    # Define number_of_circles here.
    ...

    if number_of_circles == 0:
        return background

    if ratio <= 0:
        return background

    if ratio > 1:
        ratio = 1

    largest_circle_diameter = ...
    largest_circle_radius = ...

    radius_step = ... / number_of_circles

    layers = empty_graphic()
    current_radius = largest_circle_radius

    for i in range(number_of_circles):
        # Use i to choose the color pair.
        # Update the current_radius with radius_step
        # Hint: same as in the fixed-step composition function!
        ...
        ...
        ...

        layer = split_circle(current_radius, left_color, right_color)
        layers = overlay(..., ...)
        current_radius = ...

    # Overlay the layers and the background
    return ...

show_graphic(composition(500, color_pairs, 0.5))


From what we can see, `0.5` seems to be a pretty spot on value for the `ratio` when it comes to replicating **The Swan**.

## Test the Improved Design System

Now try the same larger list again. This time, the circles should stay inside the chosen canvas proportion because the inward step adapts.

<Task>
Run the improved version with `many_color_pairs`.

Then try changing `ratio` to `0.4`, `0.75`, and `1.0`.

What changes? What stays the same?
</Task>


In [ ]:
show_graphic(composition(500, many_color_pairs, 0.75))

## Self-Check: What Did We Actually Abstract?

<DoNow id="abstraction-layers-check">
Match each abstraction to its job:

1. `split_circle`  
2. `color_pairs`  
3. `for i in range(...)`  
4. `ratio`

Jobs:

A. controls the proportion of the largest layer compared to the canvas  
B. stores the sequence of color choices  
C. repeats the construction rule for every layer  
D. hides the semicircle-building details behind one name
</DoNow>


## Bonus Sandbox: Controlled Surprise

Now that the structure is solid, we can play. This version keeps the same proportional design system, but lets the computer randomly choose color pairs.

This is where artist control meets algorithmic surprise. Sometimes it looks cool. Other times not so much.

<Task>
Run the cell several times.

Then change `number_of_circles` and `ratio`.

Question: when does randomness make the artwork more interesting, and when does it become visual soup?
</Task>


In [ ]:
from random import choice

CIRCLE_COLORS = [OFFWHITE, BLACK, BLUE, YELLOW, PINK]

def random_composition(size: float, number_of_circles: int, ratio: float) -> Graphic:
    random_pairs: list[tuple[Color, Color]] = []

    for i in range(number_of_circles):
        left_color = choice(CIRCLE_COLORS)
        right_color = choice(CIRCLE_COLORS)
        random_pairs.append((left_color, right_color))

    return composition(size, random_pairs, ratio)

show_graphic(random_composition(500, 6, 0.75))


## Final Reflection

<DoNow id="final-repetition-reflection">
Complete this sentence:

> Repetition becomes a design system when...

Try to mention both the artwork and the program.
</DoNow>

> Repetition becomes a design system when we stop copying individual parts and start describing the rule.

Next lesson, we'll expand on repeated structures with transformations and symmetry.
